In [1]:
import os
print(os.getcwd())


c:\Users\Varshitha Samiappan\StreamScope-Netflix-Analyzer\notebooks


In [2]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/processed/netflix_cleaned.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
# Extract numeric value from duration column
df["duration_numeric"] = df["duration"].str.extract(r"(\d+)").astype(float)

# Create duration_minutes for Movies
df["duration_minutes"] = np.where(
    df["type"] == "Movie",
    df["duration_numeric"],
    np.nan
)

# Create seasons column for TV Shows
df["seasons"] = np.where(
    df["type"] == "TV Show",
    df["duration_numeric"],
    np.nan
)

df[["type", "duration", "duration_minutes", "seasons"]].head(10)

,type,duration,duration_minutes,seasons
0,Movie,90 min,90.0,NaN
1,TV Show,2 Seasons,NaN,2.0
2,TV Show,1 Season,NaN,1.0
3,TV Show,1 Season,NaN,1.0
4,TV Show,2 Seasons,NaN,2.0
5,TV Show,1 Season,NaN,1.0
6,Movie,91 min,91.0,NaN
7,Movie,125 min,125.0,NaN
8,TV Show,9 Seasons,NaN,9.0
9,Movie,104 min,104.0,NaN


In [4]:
from sklearn.preprocessing import LabelEncoder

# Fill missing ratings
df["rating"] = df["rating"].fillna("Unknown")

# Encode rating
rating_encoder = LabelEncoder()
df["rating_encoded"] = rating_encoder.fit_transform(df["rating"])

df[["rating", "rating_encoded"]].head()

,rating,rating_encoded
0,PG-13,7
1,TV-MA,11
2,TV-MA,11
3,TV-MA,11
4,TV-MA,11


In [5]:
df["type_encoded"] = df["type"].map({
    "Movie": 0,
    "TV Show": 1
})

df[["type", "type_encoded"]].head()

,type,type_encoded
0,Movie,0
1,TV Show,1
2,TV Show,1
3,TV Show,1
4,TV Show,1


In [6]:
# Select features for modeling
model_df = df[[
    "release_year",
    "rating_encoded",
    "duration_minutes",
    "seasons",
    "type_encoded"
]].copy()

# Replace NaN with 0
model_df = model_df.fillna(0)

model_df.head()

,release_year,rating_encoded,duration_minutes,seasons,type_encoded
0,2020,7,90.0,0.0,0
1,2021,11,0.0,2.0,1
2,2021,11,0.0,1.0,1
3,2021,11,0.0,1.0,1
4,2021,11,0.0,2.0,1


In [7]:
# Define features and target
X = model_df.drop("type_encoded", axis=1)
y = model_df["type_encoded"]

X.head()

,release_year,rating_encoded,duration_minutes,seasons
0,2020,7,90.0,0.0
1,2021,11,0.0,2.0
2,2021,11,0.0,1.0
3,2021,11,0.0,1.0
4,2021,11,0.0,2.0


Now We Start Real ML

Next step:

1️⃣ Train-Test Split
2️⃣ Random Forest Classifier
3️⃣ Accuracy evaluation
4️⃣ Feature Importance

In [8]:
##Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (7045, 4)
Test set size: (1762, 4)


In [9]:
## Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Create model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train model
rf_model.fit(X_train, y_train)

print("Model training completed ✅")

Model training completed ✅


In [10]:
##Evaluate Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predictions
y_pred = rf_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1214
           1       1.00      1.00      1.00       548

    accuracy                           1.00      1762
   macro avg       1.00      1.00      1.00      1762
weighted avg       1.00      1.00      1.00      1762


Confusion Matrix:

[[1214    0]
 [   0  548]]


In [11]:
##Feature Importance
import pandas as pd

# Get feature importance
importance = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

feature_importance_df


,Feature,Importance
3,seasons,0.496963
2,duration_minutes,0.480415
1,rating_encoded,0.021745
0,release_year,0.000877


In [12]:
##Prepare Data for Clustering
from sklearn.preprocessing import StandardScaler

# Features only (exclude target)
cluster_features = model_df.drop("type_encoded", axis=1)

# Scale features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_features)

scaled_features[:5]


array([[ 0.65993048, -1.53739993,  0.4019493 , -0.45006651],
       [ 0.77332444,  0.50028112, -1.34454411,  1.22841571],
       [ 0.77332444,  0.50028112, -1.34454411,  0.3891746 ],
       [ 0.77332444,  0.50028112, -1.34454411,  0.3891746 ],
       [ 0.77332444,  0.50028112, -1.34454411,  1.22841571]])

In [13]:
##Apply KMeans
from sklearn.cluster import KMeans

# Choose number of clusters
kmeans = KMeans(n_clusters=3, random_state=42)

# Fit model
kmeans.fit(scaled_features)

# Assign cluster labels
model_df["cluster"] = kmeans.labels_

model_df.head()

,release_year,rating_encoded,duration_minutes,seasons,type_encoded,cluster
0,2020,7,90.0,0.0,0,1
1,2021,11,0.0,2.0,1,0
2,2021,11,0.0,1.0,1,0
3,2021,11,0.0,1.0,1,0
4,2021,11,0.0,2.0,1,0


In [14]:
##Analyze Cluster Characteristics
# Add cluster column back to original df for interpretation
df["cluster"] = model_df["cluster"]

# Check cluster distribution
cluster_counts = df["cluster"].value_counts()
cluster_counts

cluster
1    5254
0    2816
2     737
Name: count, dtype: int64

In [15]:
##Understand Each Cluster
cluster_analysis = df.groupby("cluster")[[
    "release_year",
    "duration_minutes",
    "seasons"
]].mean()

cluster_analysis

,release_year,duration_minutes,seasons
cluster,,,
0,2016.901278,29.106918,1.765149
1,2015.925200,99.698210,NaN
2,1991.343284,114.317992,1.736842
